### Análise Exploratória de Dados (EDA)

O objetivo desta etapa é explorar os dados, identificar padrões relevantes e responder às seguintes perguntas de negócio e analíticas:

- Qual é a distribuição da variável-alvo?
- Quais são as principais correlações e relações entre as variáveis ​​(features)?
- Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis ​​e a modelagem?
- Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?
- Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?
- Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?
- Quais variáveis ​​anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?
- Qual é o limiar de decisão ideal para a implementação em produção?

In [0]:
dbutils.library.restartPython()

In [0]:
dados = spark.sql("select * from fraud_detection_dev.silver.fraud_data_clean")

In [0]:
display(dados)

%md
### What is the distribution of the target variable?
------------------------------------------------------
### Qual é a distribuição da variável-alvo?

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Target distribution - PySpark
# ============================================================

target_counts = (
    dados
    .groupBy("target")
    .agg(
        F.count("*").alias("count"),
        F.sum("TransactionAmt").alias("total_amount")
    )
)

total = dados.count()

target_dist = (
    target_counts
    .withColumn(
        "percentage",
        F.round((F.col("count") / F.lit(total)) * 100, 2)
    )
    .orderBy("target")
)

display(target_dist)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Apenas o resultado agregado
target_pd = target_dist.toPandas()

legitimate = target_pd.loc[target_pd["target"] == 0].iloc[0]
fraud = target_pd.loc[target_pd["target"] == 1].iloc[0]

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(10, 3.2))

# ------------------------------------------------------------
# Composition bar
# ------------------------------------------------------------

ax.barh(
    y=0,
    width=legitimate["percentage"],
    left=0,
    height=0.32,
    color="#334155"
)

ax.barh(
    y=0,
    width=fraud["percentage"],
    left=legitimate["percentage"],
    height=0.32,
    color="#E5484D"
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

# Legitimate
ax.text(
    legitimate["percentage"] / 2,
    0,
    f'{legitimate["percentage"]:.2f}%',
    ha="center",
    va="center",
    fontsize=13,
    color="white"
)

# Fraud
ax.text(
    legitimate["percentage"] - 1.5,
    0.32,
    f'{fraud["percentage"]:.2f}% Fraud',
    ha="center",
    va="bottom",
    fontsize=11,
    color="#E5484D"
)

# ------------------------------------------------------------
# Transaction counts
# ------------------------------------------------------------

ax.text(
    0,
    -0.35,
    f'{legitimate["count"]:,.0f} legitimate transactions',
    ha="left",
    va="center",
    fontsize=10,
    color="#64748B"
)

ax.text(
    100,
    -0.35,
    f'{fraud["count"]:,.0f} fraudulent transactions',
    ha="right",
    va="center",
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

ax.set_title(
    "Transaction Distribution",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.10,
    "Class distribution across the fraud detection dataset",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Minimalist formatting
# ------------------------------------------------------------

ax.set_xlim(0, 100)
ax.set_ylim(-0.65, 0.65)

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Data Quality
-----------------
### Qualidade do dado

In [0]:
from pyspark.sql import functions as F

# Get total count
total_count = dados.count()

# Calculate quality metrics for all columns
quality_metrics = []

for col_name in dados.columns:
    # Base metrics for all columns
    col_stats = dados.agg(
        F.count(F.when(F.col(col_name).isNull(), 1)).alias("null_count"),
        F.countDistinct(col_name).alias("unique_count")
    ).collect()[0]
    
    null_count = col_stats["null_count"]
    unique_count = col_stats["unique_count"]
    
    metric = {
        "column": col_name,
        "null_count": null_count,
        "null_rate": round((null_count / total_count) * 100, 2),
        "unique_count": unique_count,
        "unique_rate": round((unique_count / total_count) * 100, 2)
    }
    
    quality_metrics.append(metric)

# Convert to DataFrame for display
quality_df = spark.createDataFrame(quality_metrics)

# Order by null_rate descending to see most problematic columns first
quality_df = quality_df.orderBy(F.col("null_rate").desc())

display(quality_df)

### Outliers detection
-----------------------
### Deteccao de outliers

**Isolation Forest**

For outlier detection, the unsupervised **Isolation Forest** algorithm will be used.

This method is suitable for the dataset due to its large number of records and multiple features. Unlike univariate statistical methods such as IQR or Z-Score, Isolation Forest can identify anomalous observations by considering the combined behavior of multiple features.

The algorithm is based on the principle that anomalous observations are rarer and more distinct from the majority of the data and, therefore, tend to be **isolated with fewer splits** in randomly generated trees.

In this project, the goal is not to assume that every outlier represents fraud. Instead, outlier detection will be used as an exploratory technique to identify transactions with unusual behavior and evaluate their relationship with the `target` variable.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# ============================================================
# 1. Select numeric features
# ============================================================

exclude_cols = [
    "TransactionID",
    "target",
    "TransactionDT",
    "ingestion_timestamp",
    "dataset_type"
]

numeric_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName()
    in ["double", "float", "integer", "long", "decimal"]
    and field.name not in exclude_cols
]

# ============================================================
# 2. Fill missing values with median
# ============================================================

median_values = {}
for col in numeric_cols:
    median_val = dados.approxQuantile(col, [0.5], 0.01)[0]
    median_values[col] = median_val if median_val is not None else 0.0

dados_filled = dados
for col, median_val in median_values.items():
    dados_filled = dados_filled.fillna({col: median_val})

In [0]:
# ============================================================
# 3. Train Isolation Forest
# ============================================================

training_data = dados_filled.select(numeric_cols).toPandas()

if_model = IsolationForest(
    contamination=0.05,
    n_estimators=300,
    max_samples=256,
    random_state=42,
    n_jobs=-1
)

if_model.fit(training_data)

In [0]:
# ============================================================
# 4. Apply model using pandas UDF
# ============================================================

from pyspark.sql.functions import pandas_udf, PandasUDFType

schema = StructType([
    StructField("anomaly_score", DoubleType(), True),
    StructField("is_outlier", IntegerType(), True)
])

@pandas_udf(schema, PandasUDFType.SCALAR)
def predict_outliers(*cols):
    df = pd.DataFrame({col_name: col for col_name, col in zip(numeric_cols, cols)})
    scores = if_model.decision_function(df)
    predictions = if_model.predict(df)
    predictions = np.where(predictions == -1, 1, 0)
    return pd.DataFrame({"anomaly_score": -scores, "is_outlier": predictions})

dados_with_outliers = dados_filled.withColumn("outlier_results",predict_outliers(*numeric_cols))

dados_with_outliers_final = (
    dados_with_outliers
    .withColumn("anomaly_score", F.col("outlier_results.anomaly_score"))
    .withColumn("is_outlier", F.col("outlier_results.is_outlier"))
    .drop("outlier_results")
)

In [0]:
# ============================================================
# 5. Outlier summary
# ============================================================

total = dados_with_outliers_final.count()

outlier_summary = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
    .orderBy("is_outlier")
)

display(outlier_summary)


### Do the anomalous observations identified by Isolation Forest exhibit a higher concentration of fraud than the general population?
---------------------------------------------------------------------------------------------------------------------
### As observações consideradas anômalas pelo Isolation Forest apresentam uma concentração de fraude maior que a população geral?

In [0]:
# ============================================================
# 6. Relationship between anomalies and fraud
# ============================================================

outlier_fraud_analysis = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("target").alias("fraud_transactions"),
        F.round(F.avg("target") * 100, 2).alias("fraud_rate_pct"),
        F.round(F.avg("anomaly_score"), 4).alias("avg_anomaly_score")
    )
)

display(outlier_fraud_analysis)

# ============================================================
# 7. Update main DataFrame
# ============================================================

dados = dados_with_outliers_final

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Prepare data
# ============================================================

outlier_pd = outlier_fraud_analysis.toPandas()

outlier_pd["group"] = outlier_pd["is_outlier"].map({
    1: "Outliers",
    0: "Regular transactions"
})

# Ordenação
outlier_pd = outlier_pd.sort_values(
    "fraud_rate_pct",
    ascending=True
)

# ============================================================
# Executive visualization
# ============================================================

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(9, 4))

colors = [
    "#475569" if group == "Regular transactions" else "#E5484D"
    for group in outlier_pd["group"]
]

bars = ax.barh(
    outlier_pd["group"],
    outlier_pd["fraud_rate_pct"],
    height=0.42,
    color=colors
)

# ============================================================
# Labels
# ============================================================

for bar, (_, row) in zip(bars, outlier_pd.iterrows()):

    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    # Main metric
    ax.text(
        width + 0.35,
        y + 0.06,
        f"{row['fraud_rate_pct']:.2f}%",
        va="center",
        ha="left",
        fontsize=14,
        color="#0F172A"
    )

    # Secondary information
    ax.text(
        width + 0.35,
        y - 0.10,
        f"{row['fraud_transactions']:,.0f} frauds  •  "
        f"{row['transactions']:,.0f} transactions",
        va="center",
        ha="left",
        fontsize=9,
        color="#64748B"
    )

# ============================================================
# Title and subtitle
# ============================================================

ax.set_title(
    "Fraud Rate by Anomaly Classification",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.05,
    "Fraud incidence is substantially higher among anomalous transactions",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ============================================================
# Formatting
# ============================================================

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xlim(
    0,
    outlier_pd["fraud_rate_pct"].max() * 1.55
)

ax.grid(False)

ax.tick_params(
    axis="y",
    length=0,
    labelsize=10,
    colors="#334155"
)

ax.set_xticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Análise

Entre as transações identificadas como anômalas, **17,45% são fraudulentas**. Considerando a quantidade de registros e a natureza do problema, não será realizada a remoção desses valores, pois eles podem representar comportamentos naturais do fenômeno analisado.

Este estudo possui caráter exploratório, com o objetivo de compreender a relação entre valores discrepantes ou comportamentos anômalos nos dados e a ocorrência de transações fraudulentas. A remoção dessas observações poderia eliminar padrões relevantes para a identificação de fraudes.

------------------------------------------------------

### Analysis

Among the transactions identified as anomalous, **17.45% are fraudulent**. Considering the number of records and the nature of the problem, these observations will not be removed, as they may represent natural behaviors of the phenomenon being analyzed.

This analysis has an exploratory purpose, aiming to understand the relationship between outliers or anomalous patterns in the data and the occurrence of fraudulent transactions. Removing these observations could eliminate relevant patterns for fraud detection.

### What are the main correlations and relationships between features?
### Quais são as principais correlações e relações entre as variáveis (features)?

### What statistical tests can provide relevant insights for feature selection and modeling?
### Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis e a modelagem?

### What is the financial impact of fraud across different transaction segments?
### Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?

### Which device and identity combinations are the strongest fraud indicators?
### Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?

### What transaction amount patterns differentiate fraudulent from legitimate transactions?
### Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?

### Which anonymous variables (V1–V339) have the greatest predictive power for fraud detection?
### Quais variáveis anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?